# Hangman RNN: improvements and ablation

Every variant is played on the same engine and the same words: 2,000 games for seed 0 and 2,000 for seed 1 (standard error of one rate is about 1.1 percentage points). `data/words_test.txt` is only used here, for final reporting. Decoding and opening choices were made on a validation split of `words_train.txt` (see the README).

Run from the `notebooks/` directory. The full ablation takes roughly 15 minutes on 2 CPU cores.

In [1]:
import sys
sys.path.insert(0, "..")

import pandas as pd
from hangman import HangmanBasic, HangmanRNN, load_model
from hangman.evaluate import evaluate

Variants, from the original model to the best one:

- **decoding**: `cell` = single most probable (position, letter) cell; `sum` = letter with the largest probability summed over hidden slots; `noisy_or` = most likely to appear in at least one hidden slot.
- **opening**: `fixed` = vowel-first order; `freq` = most common letter among training words of the same length that avoid every wrong guess so far.
- **scratch** models are trained from random initialisation on simulated game states with held-out validation words (`hangman/train.py`); **v2** is v1 fine-tuned the same way with the wrong-guess input.

In [2]:
M = "../models/"
v1 = load_model(M + "model_rnn.pth")
v2 = load_model(M + "model_rnn_v2.pth")
scratch_missed = load_model(M + "ablation/v2_scratch_missed.pth")
scratch_plain = load_model(M + "ablation/v2_scratch_nomissed.pth")

variants = [
    ("Dictionary baseline", None, None, None),
    ("v1 (original)", v1, "cell", "fixed"),
    ("v1 + sum decoding", v1, "sum", "fixed"),
    ("v1 + noisy-or decoding", v1, "noisy_or", "fixed"),
    ("v1 + sum decoding + length-conditioned opening", v1, "sum", "freq"),
    ("scratch, game-state training, no missed input, cell", scratch_plain, "cell", "fixed"),
    ("scratch, game-state training, no missed input, sum", scratch_plain, "sum", "fixed"),
    ("scratch, game-state training, missed input, cell", scratch_missed, "cell", "fixed"),
    ("scratch, game-state training, missed input, sum", scratch_missed, "sum", "fixed"),
    ("scratch, game-state training, missed input, sum + freq opening", scratch_missed, "sum", "freq"),
    ("v2 (v1 fine-tuned, missed input), cell", v2, "cell", "fixed"),
    ("v2 (v1 fine-tuned, missed input), sum", v2, "sum", "fixed"),
]

In [3]:
rows = []
for name, model, decoding, opening in variants:
    player = HangmanBasic() if model is None else HangmanRNN(None, model, decoding, opening)
    r0, r1 = (evaluate(player, games=2000, seed=s, progress=False) for s in (0, 1))
    rows.append({"variant": name, "wins seed 0": r0["wins"], "wins seed 1": r1["wins"],
                 "seed 0 %": 100 * r0["success_rate"], "seed 1 %": 100 * r1["success_rate"]})
table = pd.DataFrame(rows)
table["mean %"] = (table["seed 0 %"] + table["seed 1 %"]) / 2
table["delta vs v1 (pp)"] = table["mean %"] - table.loc[1, "mean %"]
table.round(2)

,variant,wins seed 0,wins seed 1,seed 0 %,seed 1 %,mean %,delta vs v1 (pp)
0,Dictionary baseline,369,391,18.45,19.55,19.00,-33.25
1,v1 (original),1043,1047,52.15,52.35,52.25,0.00
2,v1 + sum decoding,1086,1084,54.30,54.20,54.25,2.00
3,v1 + noisy-or decoding,1092,1070,54.60,53.50,54.05,1.80
4,v1 + sum decoding + length-conditioned opening,1079,1082,53.95,54.10,54.02,1.77
5,"scratch, game-state training, no missed input,...",929,934,46.45,46.70,46.58,-5.67
6,"scratch, game-state training, no missed input,...",1013,1014,50.65,50.70,50.68,-1.58
7,"scratch, game-state training, missed input, cell",934,941,46.70,47.05,46.88,-5.38
8,"scratch, game-state training, missed input, sum",1005,1022,50.25,51.10,50.68,-1.58
9,"scratch, game-state training, missed input, su...",1004,1023,50.20,51.15,50.68,-1.58


## Reading the table

Rows 2 to 5 change only how the original v1 network is used (no retraining). The scratch rows retrain with a different data distribution and loss; comparing "no missed input" with "missed input" isolates the wrong-guess input. The last rows warm-start v2 from v1.